# Structured generation: step by step

In [ ]:
from outlines import models, generate
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from enum import Enum
import pandas as pd
import json
from openai import OpenAI
from rich.progress import track

In [ ]:
load_dotenv()

## Define prompt

In [ ]:
country_codes = pd.read_csv("../country_codes_mapping.csv").loc[:, ["alpha-3"]].to_numpy().ravel()
CountriesEnum: Enum = Enum("Countries", {code: code for code in country_codes})

In [ ]:
class OlympicMedals(BaseModel):
    country: CountriesEnum
    year: int
    location: str
    gold: int
    silver: int
    bronze: int

In [ ]:
prompt = f"""
You are a helpful assistant who outputs a valid JSON from a given text about olympic medals.    
You MUST follow the given JSON schema: 
{OlympicMedals.model_json_schema()}

Example:
User: "France has won 16 gold medals, 26 silver medals, and 22 bronze medals during the 2024 Paris Olympic games"
Result: {OlympicMedals(country=CountriesEnum.FRA, year=2024, location="Paris", gold=16, silver=26, bronze=22).model_dump_json()}
"""

In [ ]:
print(prompt)

## Generate JSON from text

### Without outlines

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_json_response_without_outlines(user_request: str, client: OpenAI = client):
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": prompt},
            {
                "role": "user",
                "content": user_request,
            },
        ],
    )

    return completion.choices[0].message.content

### With outlines

In [ ]:
model = models.openai(
    "gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)
generator = generate.json(model, OlympicMedals)

def get_json_response_with_outlines(user_request: str, generator: generate.api.SequenceGeneratorAdapter = generator):
    response: OlympicMedals = generator(user_request)
    return response.model_dump_json()

## Evaluation

In [ ]:
def is_valid_json(response: str):
    try:
        json.loads(response)
        return 1
    except ValueError:
        return 0

In [ ]:
resp1 = get_json_response_without_outlines("During the 2020 Tokyo Olympics, which actually took place in 2021 due to COVID pandemic, Canada impressed with a record 12 gold medals for its athletes, when previously it had only won 4. In addition to that, they got 23 silver, and 15 bronze medals")

In [ ]:
resp2 = get_json_response_with_outlines("During the 2020 Tokyo Olympics, which actually took place in 2021 due to COVID pandemic, Canada impressed with a record 12 gold medals for its athletes, when previously it had only won 4. In addition to that, they got 23 silver, and 15 bronze medals")

In [ ]:
df_evaluation_set = pd.read_csv("../evaluation_set.csv")
evaluation_prompts = df_evaluation_set.to_numpy().ravel()[:10]

In [ ]:
evaluation_prompts

In [ ]:
valid_json_without_outlines = 0
for req in track(evaluation_prompts, description="Processing without outlines..."):
    valid_json_without_outlines += is_valid_json(
        get_json_response_without_outlines(
            req
        )
    )
print(f"Without outlines: {100*(valid_json_without_outlines/len(evaluation_prompts)):.2f}% success rate")

In [ ]:
valid_json_with_outlines = 0
for req in track(evaluation_prompts, description="Processing with outlines..."):
    valid_json_with_outlines += is_valid_json(
        get_json_response_with_outlines(
            req
        )
    )
print(f"Without outlines: {100*(valid_json_without_outlines/len(evaluation_prompts)):.2f}% success rate")